# Un clasificador naive Bayes a mano

En [El teorema de Bayes, palabra a palabra](one_word.ipynb), el teorema convirtió la presencia de una palabra en una probabilidad de spam. Un mensaje tiene muchas palabras, y el clasificador debería usarlas todas. Hay tres problemas por el camino, y cada uno tiene una respuesta sencilla: la suposición ingenua, el suavizado y los logaritmos. Juntos forman un clasificador naive Bayes, escrito aquí en unas pocas líneas de Python y comprobado después con el de scikit-learn.

## Los mensajes, sin repeticiones

In [ ]:
from sms_spam import load_sms_spam

messages = load_sms_spam()
messages.duplicated(subset="message").sum()

np.int64(403)

403 filas repiten un mensaje que apareció antes (281 mensajes distintos aparecen más de una vez); solo «Sorry, I'll call later» aparece 30 veces. Con una división aleatoria, copias del mismo mensaje caerían tanto en el conjunto de entrenamiento como en el de prueba, y la prueba mediría en parte si el clasificador recuerda mensajes que ya ha visto. Cada mensaje se conserva una sola vez, y la división mantiene la proporción de spam en los dos conjuntos ([por qué importan ambas cosas](../../ml-fundamentals/overfitting_and_the_split.ipynb)):

In [ ]:
from sklearn.model_selection import train_test_split

unique_messages = messages.drop_duplicates(subset="message")
train, test = train_test_split(unique_messages, test_size=0.2, stratify=unique_messages["label"], random_state=42)
len(train), len(test), train["label"].eq("spam").mean().round(3)

(4136, 1035, np.float64(0.126))

## Dos palabras juntas

Con dos palabras, el teorema de Bayes necesita la probabilidad de ver *ambas* en cada clase, $P(\text{free}, \text{call} \mid \text{spam})$. Se puede contar, y comparar con el producto de las dos probabilidades por separado:

In [ ]:
import re

import pandas as pd


def words_in(message):
    return re.findall(r"[a-z0-9£]+", message.lower())


word_sets = train["message"].map(lambda message: set(words_in(message)))
is_spam = train["label"] == "spam"


def joint_and_product(first, second, class_rows):
    has_first = word_sets[class_rows].map(lambda words: first in words)
    has_second = word_sets[class_rows].map(lambda words: second in words)
    return {"P(both | class)": (has_first & has_second).mean(), "P(first | class) × P(second | class)": has_first.mean() * has_second.mean()}


pd.DataFrame({
    ("free", "call", "spam"): joint_and_product("free", "call", is_spam),
    ("free", "call", "ham"): joint_and_product("free", "call", ~is_spam),
    ("prize", "claim", "spam"): joint_and_product("prize", "claim", is_spam),
}).T.round(4)

P(both | class)  P(first | class) × P(second | class)
free  call  spam           0.0766                                0.0962
            ham            0.0025                                0.0004
prize claim spam           0.0594                                0.0144

Los productos no son las probabilidades conjuntas. En el spam, «prize» y «claim» aparecen juntas en alrededor del 6 % de los mensajes, cuatro veces más a menudo que si fueran independientes: un mensaje que ofrece un premio suele pedirte que lo reclames. «free» y «call» aparecen juntas algo menos de lo que predice la independencia en el spam, y seis veces más a menudo en el ham. Las palabras de un mensaje dependen unas de otras.

Entonces, ¿por qué no contar siempre las combinaciones? Porque con más palabras dejan de existir. Un mensaje de 20 palabras es una combinación que aparece una vez en los datos, si es que aparece, y la combinación de un mensaje nuevo no ha aparecido nunca. Ninguna cantidad de datos puede dar la probabilidad de cada combinación de palabras.

La **suposición ingenua** (*naive*) renuncia a las combinaciones: trata las palabras como independientes entre sí *dentro de cada clase*, de modo que la probabilidad de un mensaje es el producto de las probabilidades de sus palabras:

$$P(w_1, w_2, \dots, w_n \mid \text{spam}) \approx P(w_1 \mid \text{spam}) \, P(w_2 \mid \text{spam}) \cdots P(w_n \mid \text{spam})$$

Es falsa, como muestra la tabla. Pero el clasificador no necesita que las probabilidades sean correctas, solo que sean mayores para la clase correcta; el error de la suposición suele empujar a las dos clases en la misma dirección, y la comparación lo sobrevive.

## Contar palabras en cada clase

El modelo cuenta *apariciones* de palabras, no solo su presencia: un mensaje es una bolsa de palabras, cada una con el número de veces que aparece. Para cada clase, la probabilidad de una palabra es su proporción entre todas las palabras de esa clase, que no es la probabilidad de [El teorema de Bayes, palabra a palabra](one_word.ipynb), la proporción de mensajes que contienen la palabra: contar presencia lleva al `BernoulliNB` de scikit-learn, contar apariciones al `MultinomialNB`.

$$P(w \mid \text{spam}) = \frac{\text{apariciones de } w \text{ en spam}}{\text{todas las apariciones de palabras en spam}}$$

Los recuentos, solo de los mensajes de entrenamiento:

In [ ]:
from collections import Counter

word_counts = {
    label: Counter(word for message in train.loc[train["label"] == label, "message"] for word in words_in(message))
    for label in ("ham", "spam")
}
vocabulary = set(word_counts["ham"]) | set(word_counts["spam"])
{label: sum(counts.values()) for label, counts in word_counts.items()}, len(vocabulary)

({'ham': 53070, 'spam': 13213}, 7734)

In [ ]:
pd.DataFrame({label: [word for word, _ in counts.most_common(12)] for label, counts in word_counts.items()})

,ham,spam
0,i,to
1,you,a
2,to,call
3,the,you
4,a,your
5,u,for
6,and,free
7,in,the
8,me,2
9,it,now


El ham de entrenamiento tiene unas 53 000 apariciones de palabras y el spam unas 13 000, de un vocabulario de 7734 palabras distintas. Las palabras más frecuentes de las dos clases son cortas y comunes («to», «you», «a»), y las listas solo se diferencian más abajo, donde aparecen en el spam «call», «free» y «now». Lo que decide la clasificación no es lo frecuente que es una palabra, sino lo *distinta* que es su frecuencia en las dos clases: una palabra igual de común en el ham que en el spam suma más o menos lo mismo a las dos puntuaciones, y no dice nada.

## Palabras que una clase nunca ha visto

Una palabra que nunca aparece en el spam de entrenamiento recibe una probabilidad de 0 en el spam, y entonces el producto para el spam es 0, digan lo que digan las demás palabras. Una sola palabra poco habitual en un mensaje de spam descarado lo convertiría en ham seguro. La solución es el **suavizado**: hacer como si cada palabra del vocabulario se hubiera visto $\alpha$ veces más en cada clase. Con $\alpha = 1$ se llama suavizado de Laplace:

$$P(w \mid \text{spam}) = \frac{\text{apariciones de } w \text{ en spam} + \alpha}{\text{todas las apariciones de palabras en spam} + \alpha \, |V|}$$

donde $|V|$ es el número de palabras del vocabulario, para que las probabilidades sigan sumando 1. «lol», nunca vista en el spam, antes y después:

In [ ]:
def word_probability(word, label, alpha):
    total = sum(word_counts[label].values())
    return (word_counts[label][word] + alpha) / (total + alpha * len(vocabulary))


pd.DataFrame({alpha: {label: word_probability("lol", label, alpha) for label in ("ham", "spam")} for alpha in (0, 1)}).rename_axis(columns="alpha")

alpha,0,1
ham,0.001168,0.001036
spam,0.000000,0.000048


Sin suavizado, «lol» tiene una probabilidad de 0 en el spam, y un solo «lol» convertiría cualquier mensaje en ham. Con $\alpha = 1$ recibe una probabilidad pequeña en el spam, de unos 0,00005, unas veinte veces menos que en el ham: sigue siendo una prueba fuerte de ham, pero ya no un veto. El suavizado mueve probabilidad de las palabras frecuentes a las raras y a las nunca vistas: en el ham, «lol» pasa de 0,00117 a 0,00104.

## Productos de números pequeños

Cada probabilidad de palabra es pequeña, de uno entre mil o menos, y un mensaje multiplica docenas de ellas. Los números de coma flotante no pueden bajar de unos $10^{-308}$; más allá, un producto se vuelve exactamente 0:

In [ ]:
import numpy as np

probabilities = np.full(120, 1e-3)
np.prod(probabilities), np.log(probabilities).sum()

(np.float64(0.0), np.float64(-828.9306334778564))

Un producto de 120 probabilidades de uno entre mil es $10^{-360}$, guardado como 0. Su logaritmo es un número perfectamente normal, unos -829. Como el logaritmo convierte los productos en sumas y conserva el orden (el producto mayor tiene el logaritmo mayor), el clasificador compara en su lugar sumas de logaritmos:

$$\text{puntuación}(\text{spam}) = \log P(\text{spam}) + \sum_{w \in \text{mensaje}} \text{recuento}(w) \, \log P(w \mid \text{spam})$$

y predice la clase con la puntuación más alta. Las dos puntuaciones se pueden volver a convertir en probabilidades que suman 1, sin calcular nunca los productos minúsculos: $P(\text{spam} \mid \text{mensaje}) = 1 / (1 + e^{\,\text{puntuación}(\text{ham}) - \text{puntuación}(\text{spam})})$. Es la sigmoide de la [Regresión logística](../../algorithms/linear-models/logistic_regression.ipynb), aplicada a $\text{puntuación}(\text{spam}) - \text{puntuación}(\text{ham})$, una suma ponderada de los recuentos de palabras: con dos clases, naive Bayes es un clasificador lineal de la misma forma, cuyos pesos salen de contar en lugar de minimizar un error.

## El clasificador

Todo lo anterior, en una clase con las convenciones de scikit-learn: `fit` aprende de los mensajes de entrenamiento y devuelve el objeto, lo que aprende se guarda en atributos que terminan en `_`, y `predict_proba` y `predict` responden para mensajes nuevos. Las palabras que no estaban en el vocabulario de entrenamiento no aportan información sobre ninguna clase, y se ignoran:

In [ ]:
class NaiveBayesSpamFilter:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, texts, labels):
        labels = pd.Series(list(labels))
        self.classes_ = ["ham", "spam"]
        self.log_prior_ = {label: np.log((labels == label).mean()) for label in self.classes_}
        counts = {label: Counter() for label in self.classes_}
        for text, label in zip(texts, labels):
            counts[label].update(words_in(text))
        self.vocabulary_ = set(counts["ham"]) | set(counts["spam"])
        self.log_likelihood_ = {}
        for label in self.classes_:
            denominator = sum(counts[label].values()) + self.alpha * len(self.vocabulary_)
            self.log_likelihood_[label] = {
                word: np.log((counts[label][word] + self.alpha) / denominator) for word in self.vocabulary_
            }
        return self

    def log_scores(self, text):
        known_words = [word for word in words_in(text) if word in self.vocabulary_]
        return {
            label: self.log_prior_[label] + sum(self.log_likelihood_[label][word] for word in known_words)
            for label in self.classes_
        }

    def predict_proba(self, texts):
        spam_probabilities = []
        for text in texts:
            scores = self.log_scores(text)
            spam_probabilities.append(1 / (1 + np.exp(scores["ham"] - scores["spam"])))
        return np.column_stack([1 - np.array(spam_probabilities), spam_probabilities])

    def predict(self, texts):
        return np.where(self.predict_proba(texts)[:, 1] > 0.5, "spam", "ham")

In [ ]:
spam_filter = NaiveBayesSpamFilter(alpha=1.0).fit(train["message"], train["label"])

examples = [
    "Congratulations! You have won a free prize. Call now to claim it",
    "Are we still meeting for lunch tomorrow?",
    "Free entry to the party tonight, just text me when you arrive",
]
pd.DataFrame({"P(spam)": spam_filter.predict_proba(examples)[:, 1].round(4), "prediction": spam_filter.predict(examples)}, index=examples)

,P(spam),prediction
Congratulations! You have won a free prize. Call now to claim it,1.0000,spam
Are we still meeting for lunch tomorrow?,0.0000,ham
"Free entry to the party tonight, just text me when you arrive",0.2858,ham


El primer mensaje es spam descarado y el segundo ham normal, y el filtro está seguro de ambos: 1,0000 y 0,0000, con cuatro decimales. El tercero usa las palabras favoritas del spam, «free», «entry», «text», en un mensaje entre amigos, y el filtro duda: 0,29, ham. Fíjate en lo extremas que son las dos primeras probabilidades. Multiplicar docenas de probabilidades de palabras como si fueran independientes hace que naive Bayes esté mucho más seguro de lo que debería: sus probabilidades ordenan bien los mensajes, pero no deben leerse literalmente.

Con los mensajes de prueba, con las medidas de [Evaluar un clasificador](../../evaluation/classification_metrics.ipynb); «spam» es la clase positiva:

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

test_predictions = spam_filter.predict(test["message"])
pd.DataFrame(confusion_matrix(test["label"], test_predictions, labels=["ham", "spam"]),
             index=["true ham", "true spam"], columns=["predicted ham", "predicted spam"])

,predicted ham,predicted spam
true ham,900,4
true spam,10,121


In [ ]:
{
    "accuracy": (test_predictions == test["label"]).mean(),
    "precision (spam)": precision_score(test["label"], test_predictions, pos_label="spam"),
    "recall (spam)": recall_score(test["label"], test_predictions, pos_label="spam"),
}

{'accuracy': np.float64(0.9864734299516909),
 'precision (spam)': 0.968,
 'recall (spam)': 0.9236641221374046}

De los 1035 mensajes de prueba, 14 se clasifican mal: 4 mensajes de ham enviados al spam, y 10 mensajes de spam que se cuelan. Cuando el filtro dice spam acierta el 97 % de las veces, y atrapa el 92 % del spam. Para un filtro, los cuatro mensajes de ham pesan más que los diez de spam: un mensaje legítimo perdido cuesta más que un mensaje de spam borrado a mano. [El filtro de spam con scikit-learn](spam_filter_scikit_learn.ipynb) analiza ese compromiso.

## Lo mismo con scikit-learn

scikit-learn divide el trabajo en dos: `CountVectorizer` convierte cada mensaje en sus recuentos de palabras, con el mismo paso a minúsculas y el mismo patrón para las palabras, y `MultinomialNB` es el clasificador de arriba, suavizado incluido. Ajustados con los mismos mensajes de entrenamiento, deberían dar las mismas probabilidades:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

library_filter = make_pipeline(CountVectorizer(token_pattern=r"[a-z0-9£]+"), MultinomialNB(alpha=1.0))
library_filter.fit(train["message"], train["label"])
library_probabilities = library_filter.predict_proba(test["message"])[:, 1]
our_probabilities = spam_filter.predict_proba(test["message"])[:, 1]
{
    "same predictions": (library_filter.predict(test["message"]) == test_predictions).mean(),
    "largest difference in P(spam)": np.abs(library_probabilities - our_probabilities).max(),
}

{'same predictions': np.float64(1.0),
 'largest difference in P(spam)': np.float64(2.6756374893466273e-14)}

Todas las predicciones son iguales, y las probabilidades difieren en menos de $10^{-13}$, el error de redondeo de la aritmética de coma flotante: unas pocas líneas de Python y el `MultinomialNB` de scikit-learn son el mismo modelo.

[El filtro de spam con scikit-learn](spam_filter_scikit_learn.ipynb) parte de ahí: otras formas de convertir texto en números, una comparación con otro clasificador, los costes de cada tipo de error, y en qué se equivoca el filtro.